# Feature Engineering

---

1. Import packages
2. Load data
3. Feature engineering

---

## 1. Import packages

In [3]:
import pandas as pd

---
## 2. Load data

In [4]:
df = pd.read_csv("/content/clean_data_after_eda.csv")
df["date_activ"] = pd.to_datetime(df["date_activ"], format='%Y-%m-%d')
df["date_end"] = pd.to_datetime(df["date_end"], format='%Y-%m-%d')
df["date_modif_prod"] = pd.to_datetime(df["date_modif_prod"], format='%Y-%m-%d')
df["date_renewal"] = pd.to_datetime(df["date_renewal"], format='%Y-%m-%d')

In [5]:
df.head(3)

,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,...,var_6m_price_off_peak_var,var_6m_price_peak_var,var_6m_price_mid_peak_var,var_6m_price_off_peak_fix,var_6m_price_peak_fix,var_6m_price_mid_peak_fix,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak,churn
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,...,0.000131,4.100838e-05,0.000908,2.086294,99.530517,44.235794,2.086425,9.953056e+01,44.236702,1
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,...,0.000003,1.217891e-03,0.000000,0.009482,0.000000,0.000000,0.009485,1.217891e-03,0.000000,0
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,...,0.000004,9.450150e-08,0.000000,0.000000,0.000000,0.000000,0.000004,9.450150e-08,0.000000,0


---

## 3. Feature engineering

### Difference between off-peak prices in December and preceding January

Below is the code created by your colleague to calculate the feature described above. Use this code to re-create this feature and then think about ways to build on this feature to create features with a higher predictive power.

In [6]:
price_df = pd.read_csv('/content/price_data (1).csv')
price_df["price_date"] = pd.to_datetime(price_df["price_date"], format='%Y-%m-%d')
price_df.head()

,id,price_date,price_off_peak_var,price_peak_var,price_mid_peak_var,price_off_peak_fix,price_peak_fix,price_mid_peak_fix
0,038af19179925da21a25619c5a24b745,2015-01-01,0.151367,0.0,0.0,44.266931,0.0,0.0
1,038af19179925da21a25619c5a24b745,2015-02-01,0.151367,0.0,0.0,44.266931,0.0,0.0
2,038af19179925da21a25619c5a24b745,2015-03-01,0.151367,0.0,0.0,44.266931,0.0,0.0
3,038af19179925da21a25619c5a24b745,2015-04-01,0.149626,0.0,0.0,44.266931,0.0,0.0
4,038af19179925da21a25619c5a24b745,2015-05-01,0.149626,0.0,0.0,44.266931,0.0,0.0


In [7]:
# Group off-peak prices by companies and month
monthly_price_by_id = price_df.groupby(['id', 'price_date']).agg({'price_off_peak_var': 'mean', 'price_off_peak_fix': 'mean'}).reset_index()

# Get january and december prices
jan_prices = monthly_price_by_id.groupby('id').first().reset_index()
dec_prices = monthly_price_by_id.groupby('id').last().reset_index()

# Calculate the difference
diff = pd.merge(dec_prices.rename(columns={'price_off_peak_var': 'dec_1', 'price_off_peak_fix': 'dec_2'}), jan_prices.drop(columns='price_date'), on='id')
diff['offpeak_diff_dec_january_energy'] = diff['dec_1'] - diff['price_off_peak_var']
diff['offpeak_diff_dec_january_power'] = diff['dec_2'] - diff['price_off_peak_fix']
diff = diff[['id', 'offpeak_diff_dec_january_energy','offpeak_diff_dec_january_power']]
diff.head()

,id,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power
0,0002203ffbb812588b632b9e628cc38d,-0.006192,0.162916
1,0004351ebdd665e6ee664792efc4fd13,-0.004104,0.177779
2,0010bcc39e42b3c2131ed2ce55246e3c,0.050443,1.500000
3,0010ee3855fdea87602a5b7aba8e42de,-0.010018,0.162916
4,00114d74e963e47177db89bc70108537,-0.003994,-0.000001


Now it is time to get creative and to conduct some of your own feature engineering! Have fun with it, explore different ideas and try to create as many as you can!

In [8]:
price_features = price_df.groupby("id").agg(
    avg_off_peack_var=("price_off_peak_var", "mean"),
    avg_peak_var=("price_peak_var", "mean"),
    avg_mid_peak_fix=("price_mid_peak_fix", "mean"),
    std_off_peak_fix=("price_off_peak_fix", "std"), # Corrected column and function
    std_peak_var=("price_peak_var", "std"),       # Corrected column and function
    std_mid_peak_fix=("price_mid_peak_fix", "std"), # Corrected column and function
).reset_index()
price_features.head()

,id,avg_off_peack_var,avg_peak_var,avg_mid_peak_fix,std_off_peak_fix,std_peak_var,std_mid_peak_fix
0,0002203ffbb812588b632b9e628cc38d,0.124338,0.103794,16.280694,6.341481e-02,0.001989,0.025366
1,0004351ebdd665e6ee664792efc4fd13,0.146426,0.000000,0.000000,8.753223e-02,0.000000,0.000000
2,0010bcc39e42b3c2131ed2ce55246e3c,0.181558,0.000000,0.000000,7.723930e-01,0.000000,0.000000
3,0010ee3855fdea87602a5b7aba8e42de,0.118757,0.098292,16.258971,8.507958e-02,0.002580,0.034032
4,00114d74e963e47177db89bc70108537,0.147926,0.000000,0.000000,5.908392e-07,0.000000,0.000000


In [9]:
diff = diff.merge(price_features,on='id', how= 'left')
diff.head()

,id,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power,avg_off_peack_var,avg_peak_var,avg_mid_peak_fix,std_off_peak_fix,std_peak_var,std_mid_peak_fix
0,0002203ffbb812588b632b9e628cc38d,-0.006192,0.162916,0.124338,0.103794,16.280694,6.341481e-02,0.001989,0.025366
1,0004351ebdd665e6ee664792efc4fd13,-0.004104,0.177779,0.146426,0.000000,0.000000,8.753223e-02,0.000000,0.000000
2,0010bcc39e42b3c2131ed2ce55246e3c,0.050443,1.500000,0.181558,0.000000,0.000000,7.723930e-01,0.000000,0.000000
3,0010ee3855fdea87602a5b7aba8e42de,-0.010018,0.162916,0.118757,0.098292,16.258971,8.507958e-02,0.002580,0.034032
4,00114d74e963e47177db89bc70108537,-0.003994,-0.000001,0.147926,0.000000,0.000000,5.908392e-07,0.000000,0.000000


In [10]:
diff['price-range'] = (
    diff['avg_peak_var'] -
    diff['avg_off_peack_var']
)

diff[['id', 'avg_off_peack_var',
      'avg_peak_var',
      'price-range']].head()

,id,avg_off_peack_var,avg_peak_var,price-range
0,0002203ffbb812588b632b9e628cc38d,0.124338,0.103794,-0.020545
1,0004351ebdd665e6ee664792efc4fd13,0.146426,0.000000,-0.146426
2,0010bcc39e42b3c2131ed2ce55246e3c,0.181558,0.000000,-0.181558
3,0010ee3855fdea87602a5b7aba8e42de,0.118757,0.098292,-0.020465
4,00114d74e963e47177db89bc70108537,0.147926,0.000000,-0.147926


In [11]:
diff[['offpeak_diff_dec_january_energy', 'offpeak_diff_dec_january_power']].head()

,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power
0,-0.006192,0.162916
1,-0.004104,0.177779
2,0.050443,1.500000
3,-0.010018,0.162916
4,-0.003994,-0.000001


In [13]:
price_df['peak_offpeak_diff'] = (
    price_df['price_peak_var'] -
    price_df['price_off_peak_var']
)

price_df[['price_peak_var',
          'price_off_peak_var',
          'price_off_peak_var']].head()

,price_peak_var,price_off_peak_var,price_off_peak_var
0,0.0,0.151367,0.151367
1,0.0,0.151367,0.151367
2,0.0,0.151367,0.151367
3,0.0,0.149626,0.149626
4,0.0,0.149626,0.149626


In [17]:
df['consumption_diffrence'] = (
    df['cons_12m'] -
    (df['cons_last_month'] * 12)
)

df[['cons_12m',
'cons_last_month',
'consumption_diffrence']] . head()

,cons_12m,cons_last_month,consumption_diffrence
0,0,0,0
1,4660,0,4660
2,544,0,544
3,1584,0,1584
4,4425,526,-1887


In [20]:
df['consumption_ratio'] = (
    df['cons_last_month'] /
    (df['cons_12m'] + 1e-6)
)

df[['cons_12m',
'cons_last_month',
'consumption_ratio']] .head()

,cons_12m,cons_last_month,consumption_ratio
0,0,0,0.00000
1,4660,0,0.00000
2,544,0,0.00000
3,1584,0,0.00000
4,4425,526,0.11887
